In [1]:
import random
from collections import deque

m,n = map(int,input("Nhap so dong va cot: ").split())
room = []

#Vi tri thuc te
real_x = random.randint(0, m-1)
real_y = random.randint(0, n-1)

for i in range(m):
    row = [random.randint(0,1) for _ in range(n)]
    room.append(row)

room[real_x][real_y] = 0

def print_room(room_state,real_x,real_y,belief_posistions=None):
    for i in range(m):
        for j in range(n):
            if i == real_x and j == real_y:
                char = "M"
            else:
                char = str(room_state[i][j])

            if belief_posistions and (i,j) in belief_posistions:
                print(f"[{char}]", end=" ")
            else:
                print(f"{char}", end=" ")
        print()

print(f"Vi tri thuc te ban dau {real_x}, {real_y}")
print_room(room,real_x,real_y)

def is_clean(room_state):
    for row in room_state:
        if 1 in row:
            return False
    return True

def transition_belief(current_room, belief_posistions, action):
    next_posistions = set()
    next_room = [list(row) for row in current_room]

    for (x, y) in belief_posistions:
        if action == "UP":     nx, ny = max(0, x - 1), y
        elif action == "DOWN": nx, ny = min(m - 1, x + 1), y
        elif action == "LEFT": nx, ny = x, max(0, y - 1)
        elif action == "RIGHT":nx, ny = x, min(n - 1, y + 1)

        next_posistions.add((nx, ny))

    if len(next_posistions) == 1:
        (only_x, only_y) = list(next_posistions)[0]
        next_room[only_x][only_y] = 0

    next_room_tuple = tuple(tuple(row) for row in next_room)
    next_posistions_frozenset = frozenset(next_posistions)

    return next_room_tuple, next_posistions_frozenset

def sensorless_search(start_room):
    start_room_tuple = tuple(tuple(row) for row in start_room)

    initial_posistions = frozenset((i,j) for i in range(m) for j in range(n))
    start_room_list = [list(row) for row in start_room_tuple]
    for (i,j) in initial_posistions:
        pass
    frontier = deque([(start_room_tuple,initial_posistions,[])])
    reached = set()

    while frontier:
        curr_room, curr_beliefs,path = frontier.popleft()

        if is_clean(curr_room):
            return path, reached

        signature = (curr_room, curr_beliefs)
        if signature in reached:
            continue
        reached.add(signature)

        for action in ["UP","DOWN","LEFT","RIGHT"]:
            next_room, next_beliefs = transition_belief(curr_room,curr_beliefs,action)
            child_signature = (next_room, next_beliefs)

            if child_signature not in reached:
                frontier.append((next_room, next_beliefs, path + [action]))

    return None, reached

actions, reached = sensorless_search(room)

print("\n------ Tìm kiếm không có quan sát ------")

if actions is None:
    print("Không tìm thấy chuỗi hành động mù nào để làm sạch phòng!")
else:
    print(f"Tìm thấy giải pháp mù! Tổng số bước: {len(actions)}")
    print("Chuỗi hành động cố định:", " -> ".join(actions))
    print(f"Số trạng thái niềm tin (Belief States) đã duyệt: {len(reached)}")

    print("\n--- MÔ PHỎNG QUÁ TRÌNH CHẠY THỰC TẾ ---")
    print("Ký hiệu: M là vị trí THỰC, [...] là các vị trí robot NGHĨ mình có thể ở đó.")
    print("Nếu ô vừa có M vừa có [...] thì hiển thị là [M].")

    curr_room_state = [list(row) for row in room]
    curr_real_x, curr_real_y = real_x, real_y

    curr_beliefs = frozenset((i, j) for i in range(m) for j in range(n))

    def print_correct_room(room_state, rx, ry, beliefs):
        for i in range(m):
            for j in range(n):
                is_real = (i == rx and j == ry)
                is_believed = ((i, j) in beliefs)
                val = room_state[i][j]

                if is_real and is_believed:
                    print(f"[{val if val != 0 else 'M'}]", end=" ")
                elif is_real:
                    print(f" M ", end=" ")
                elif is_believed:
                    print(f"[{val}]", end=" ")
                else:
                    print(f" {val} ", end=" ")
            print()

    print("Trạng thái bắt đầu:")
    print_correct_room(curr_room_state, curr_real_x, curr_real_y, curr_beliefs)
    print("-" * 35)

    for idx, act in enumerate(actions, 1):
        print(f"Bước {idx}: Thực hiện hành động [{act}]")

        if act == "UP":
            curr_real_x = max(0, curr_real_x - 1)
        elif act == "DOWN":
            curr_real_x = min(m - 1, curr_real_x + 1)
        elif act == "LEFT":
            curr_real_y = max(0, curr_real_y - 1)
        elif act == "RIGHT":
            curr_real_y = min(n - 1, curr_real_y + 1)

        curr_room_state[curr_real_x][curr_real_y] = 0

        temp_room = [list(row) for row in curr_room_state]
        next_beliefs = set()
        for (x, y) in curr_beliefs:
            if act == "UP": nx, ny = max(0, x - 1), y
            elif act == "DOWN": nx, ny = min(m - 1, x + 1), y
            elif act == "LEFT": nx, ny = x, max(0, y - 1)
            elif act == "RIGHT": nx, ny = x, min(n - 1, y + 1)
            next_beliefs.add((nx, ny))

        curr_beliefs = frozenset(next_beliefs)

        print_correct_room(curr_room_state, curr_real_x, curr_real_y, curr_beliefs)
        print(f"Robot đoán mình có thể đang ở 1 trong {len(curr_beliefs)} ô vuông.")
        print("-" * 35)

Vi tri thuc te ban dau 1, 2
1 1 1 
1 1 M 
1 1 1 

------ ÁP DỤNG THUẬT TOÁN SENSORLESS SEARCH ------
Tìm thấy giải pháp mù! Tổng số bước: 12
Chuỗi hành động cố định: UP -> UP -> LEFT -> LEFT -> DOWN -> DOWN -> RIGHT -> UP -> UP -> RIGHT -> DOWN -> DOWN
Số trạng thái niềm tin (Belief States) đã duyệt: 703

--- MÔ PHỎNG QUÁ TRÌNH CHẠY THỰC TẾ ---
Ký hiệu: M là vị trí THỰC, [...] là các vị trí robot NGHĨ mình có thể ở đó.
Nếu ô vừa có M vừa có [...] thì hiển thị là [M].
Trạng thái bắt đầu:
[1] [1] [1] 
[1] [1] [M] 
[1] [1] [1] 
-----------------------------------
Bước 1: Thực hiện hành động [UP]
[1] [1] [M] 
[1] [1] [0] 
 1   1   1  
Robot đoán mình có thể đang ở 1 trong 6 ô vuông.
-----------------------------------
Bước 2: Thực hiện hành động [UP]
[1] [1] [M] 
 1   1   0  
 1   1   1  
Robot đoán mình có thể đang ở 1 trong 3 ô vuông.
-----------------------------------
Bước 3: Thực hiện hành động [LEFT]
[1] [M]  0  
 1   1   0  
 1   1   1  
Robot đoán mình có thể đang ở 1 trong 2 ô vuô